# 파레토 2단계 탐색 · 추천 없음과 두 AI 비교

1단계 λ=0·0.5·1 학습 → 동결 검증으로 구간 선택 → 2단계 집중 학습 → 별도 동결 검증·최종 평가. 세 방식의 결정시간 공식은 동일합니다. 기존 결과는 보존하고 새 실험 ID를 만듭니다. 기본 pilot은 5일 점검 실행입니다. GitHub의 지정 커밋을 불러오며 결과·계획·체크포인트는 Drive에 저장합니다.


## 1. GitHub 버전

공개 저장소 읽기에는 토큰이 필요 없습니다. 비공개 저장소만 Colab Secrets에 `KIOSK_GITHUB_TOKEN`을 등록하세요. 재개 시 `REVISION`에 progress.json의 git_commit을 입력하세요.

In [ ]:
import os, re, sys, subprocess, tempfile
from pathlib import Path
REPOSITORY = "phj9187/koisk-simulation" #@param {type:"string"}
REVISION = "main" #@param {type:"string"}
if not re.fullmatch(r"[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+", REPOSITORY):
    raise ValueError("저장소는 owner/name 형식이어야 합니다.")
if not re.fullmatch(r"[A-Za-z0-9_./-]+", REVISION) or REVISION.startswith("-"):
    raise ValueError("브랜치 또는 커밋 해시를 입력하세요.")
if any(n.startswith(("kiosk_research", "kiosk_sim", "kiosk_cloud")) for n in sys.modules):
    raise RuntimeError("이전 코드가 로드되어 있습니다. 런타임을 다시 시작한 뒤 실행하세요.")
token = None
try:
    from google.colab import userdata
    token = userdata.get("KIOSK_GITHUB_TOKEN")
except Exception:
    pass
CODE_ROOT = Path(tempfile.mkdtemp(prefix="kiosk-research-", dir="/content"))
with tempfile.TemporaryDirectory() as auth_dir:
    helper = Path(auth_dir) / "askpass.py"
    helper.write_text("#!/usr/bin/env python3\nimport os,sys\nprint('x-access-token' if 'username' in sys.argv[1].lower() else os.environ.get('KIOSK_EPHEMERAL_TOKEN',''))\n")
    helper.chmod(0o700)
    child_env = dict(os.environ, GIT_TERMINAL_PROMPT="0", GIT_ASKPASS=str(helper))
    if token:
        child_env["KIOSK_EPHEMERAL_TOKEN"] = token
    try:
        commands = [
            ["git", "init", str(CODE_ROOT)],
            ["git", "-C", str(CODE_ROOT), "remote", "add", "origin", "https://github.com/"+REPOSITORY+".git"],
            ["git", "-C", str(CODE_ROOT), "fetch", "--depth", "1", "origin", REVISION],
            ["git", "-C", str(CODE_ROOT), "checkout", "--detach", "FETCH_HEAD"],
        ]
        for command in commands:
            result = subprocess.run(command, env=child_env, capture_output=True, text=True)
            if result.returncode:
                raise RuntimeError("GitHub 읽기 실패. 저장소·버전·비공개 저장소 권한을 확인하세요. 토큰은 출력하지 않습니다.")
    finally:
        child_env.pop("KIOSK_EPHEMERAL_TOKEN", None)
        token = None
COMMIT = subprocess.check_output(["git","-C",str(CODE_ROOT),"rev-parse","HEAD"],text=True).strip()
subprocess.check_call([sys.executable,"-m","pip","install","-q","-r",str(CODE_ROOT/"requirements.txt")])
sys.path.insert(0,str(CODE_ROOT/"src"))
print("코드 경로:", CODE_ROOT)
print("실행 커밋:", COMMIT)


## 2. Drive 연결

처음 인증 요청은 직접 승인하세요. 공유 PC에서는 본인 계정인지 확인하세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from pathlib import Path
DATA_ROOT = Path("/content/drive/MyDrive/AI-Kiosk-Data")
DATA_ROOT.mkdir(parents=True, exist_ok=True)


## 3. 자동 2단계 탐색 · 실시간 평가 그래프

pilot: 8매장·1평가기준·5고객, 1일 넓은 학습/1일 구간 선택 검증/1일 집중 학습/1일 최종 검증/1일 평가(총 5일). research: 9평가기준, 10/5/10/5/5일(35일) × 5 독립 반복. continual: 같은 35일 주기 사이에 신경망을 이어 학습합니다. α·β는 각 기준 안에서 고정합니다. 두 AI의 후보를 함께 보고 기준별 공통 λ 구간을 정하며, 무릎점이 불명확하면 전체 범위를 유지합니다. 집중 학습의 매 5번째 날짜에는 0·0.5·1을 재확인합니다. 추천 없음 대비 개선량은 별도 최종 평가 날짜에서 계산합니다. 중지 시각은 선택 사항이며 별도의 일수 상한은 없습니다.


In [ ]:
from datetime import datetime,timezone
from kiosk_research.config import Config
from kiosk_research.live import run_live
PRESET = "pilot" #@param ["pilot", "research", "continual"]
DEADLINE_KST = "" #@param {type:"string"}
RESUME_ID = "" #@param {type:"string"}
deadline = datetime.fromisoformat(DEADLINE_KST) if DEADLINE_KST.strip() else None
if deadline is not None and (deadline.tzinfo is None or deadline <= datetime.now(timezone.utc)):
    raise ValueError("미래의 시간대 포함 중지 시각을 입력하세요.")
config = Config.load(CODE_ROOT / "config" / ("actor_critic_"+PRESET+".yaml"))
if deadline is not None: config.early_stopping = False
config.auto_cleanup = False
assert 'no_recommendation' in config.models and config.decision_time_mode == 'equal'
assert config.lambda_search == 'pareto_two_stage'
run = run_live(DATA_ROOT, config, deadline.astimezone(timezone.utc).isoformat() if deadline else None,
               resume_id=RESUME_ID.strip() or None, target_day=None)
print("Saved:",run.id,run.state["day"],run.state["stop_reason"])


## 4. 진행 상황·저장 결과

이 셀은 실행 완료 후 자동으로 표시됩니다. 실행 중에는 Drive의 latest_run.json 또는 experiments/실험ID/progress.json에서 마지막 갱신 시각·완료 날짜·체크포인트·완료 매장 수를 확인하세요. 파일의 running 표시가 오래됐다면 현재 런타임이 살아 있다는 보장은 아닙니다.

In [ ]:
import json
from kiosk_research.session import status
print(json.dumps(status(DATA_ROOT), ensure_ascii=False, indent=2))
from kiosk_research.persistence import read_json
print(read_json(run.directory / "session_result.json"))


## 5. 선택: GitHub 요약 업로드

아래 설정을 true로 바꾼 경우에만 Colab Secrets의 쓰기 권한 토큰을 사용합니다. 원시 자료·체크포인트는 Drive에 남고 작은 요약만 새로운 results 브랜치에 올라갑니다. 이미 같은 요약을 올렸으면 저장된 업로드 기록을 보여줍니다.

In [ ]:
UPLOAD_SUMMARY = False #@param {type:"boolean"}
if UPLOAD_SUMMARY:
    from google.colab import userdata
    from kiosk_research.github import upload_summary
    summary = Path(read_json(run.directory / "session_result.json")["summary"])
    if (summary / "github_upload.json").exists():
        print(read_json(summary / "github_upload.json"))
    else:
        print(upload_summary(summary, REPOSITORY, userdata.get("KIOSK_GITHUB_TOKEN")))
